In [1]:
import numpy as np

In [2]:
# Speed of light
c = 2.99792458e10      # cm/s

# Electron charge (CGS ESU)
e = 4.803204712e-10    # statcoulomb (esu)

# Electron mass
m_e = 9.10938356e-28    # g

# Proton mass
m_p = 1.67262192369e-24 # g

# Boltzmann constant
k_B = 1.380649e-16      # erg/K

# Parsec
p_c = 3.085677581e18    # cm

# Pi
pi = np.pi

In [3]:
alpha = 0.26

S_nu_Jy = 0.66

nu = 1.0e9          # Hz

distance_kpc = 50.0

theta_arcmin = 3.8

f = 0.25

vs_kms = 170.0

cs_kms = 10.0

xi = 4.0

beta = 1.0

In [4]:
# Flux density
S_nu = S_nu_Jy * 1e-23      # erg s^-1 cm^-2 Hz^-1

# Distance
d = distance_kpc * 1e3 * p_c

# Angular radius
theta = theta_arcmin * np.pi / (180.0 * 60.0)

# Velocities
vs = vs_kms * 1e5           # cm/s

cs = cs_kms * 1e5           # cm/s

In [5]:
print("S_nu =", S_nu)
print("d =", d)
print("theta =", theta)
print("vs =", vs)
print("cs =", cs)


S_nu = 6.6000000000000004e-24
d = 1.5428387905e+23
theta = 0.001105375192929742
vs = 17000000.0
cs = 1000000.0


In [6]:
Gamma = 2*alpha + 1

M1 = vs/cs

gamma = 5.0/3.0

sigma = (
    (gamma + 1)*M1**2
    /
    (2 + (gamma - 1)*M1**2)
)

kappa = 1.0 / np.sqrt(
    (1 + 2*sigma**2)/3
)

print("Gamma =", Gamma)
print("M1 =", M1)
print("sigma =", sigma)
print("kappa =", kappa)

Gamma = 1.52
M1 = 17.0
sigma = 3.958904109589041
kappa = 0.3045449391457259


In [7]:
from scipy.special import gamma as GammaFunc

In [8]:
pitch_avg = (
    np.sqrt(np.pi)/2
    *
    GammaFunc((Gamma + 5)/4)
    /
    GammaFunc((Gamma + 7)/4)
)

print("pitch_avg =", pitch_avg)

pitch_avg = 0.7486537278187587


In [9]:
c1 = (
    3*e
    /
    (4*pi*m_e**3*c**5)
)

print("c1 =", c1)

c1 = 6.264292181290573e+18


In [10]:
c5 = (
    np.sqrt(3)
    * e**3
    /
    (16*pi*m_e*c**2)
)

c5 *= (
    GammaFunc((3*Gamma - 1)/12)
    *
    GammaFunc((3*Gamma + 7)/12)
)

c5 *= (
    2/(Gamma + 1)
)

print("c5 =", c5)

c5 = 1.1456791163099356e-23


In [11]:
J = (
    S_nu
    /
    (
        (4*pi/3)
        * f
        * theta**3
        * d
        * c5
        * pitch_avg
        * (nu/(2*c1))**((1-Gamma)/2)
    )
)

print("J =", J)

J = 8.353473860768132e-18


In [12]:
from scipy.integrate import quad
from scipy.optimize import root_scalar

In [13]:
eta = (
    4/np.sqrt(np.pi)
    *
    (1/(Gamma - 1))
    *
    xi**3
    *
    np.exp(-xi**2)
)

print("eta =", eta)

eta = 3.125719303844076e-05


In [14]:
Tp = (
    (
        gamma
        + 1
        + 2*gamma*(M1**2 - 1)
    )
    *
    (
        2 + (gamma - 1)*M1**2
    )
    /
    (
        gamma*(gamma+1)**2*M1**2
    )
)

Tp *= m_p*cs**2/k_B

print("Tp =", Tp, "K")

Tp = 662823.6133350248 K


In [15]:
Te = beta*Tp

print("Te =", Te)

Te = 662823.6133350248


In [16]:
def I1_of_B(B):

    pinj = xi*np.sqrt(2*m_e*k_B*Te)

    Einf = (
        (
            (sigma - 1)
            /
            (3*sigma)
        )
        *
        (
            m_e**2*c**3*vs
        )
        /
        (
            (1 + np.sqrt(kappa))
            *
            np.sqrt((2/27)*e**3*B)
        )
    )

    xinj = pinj/(m_e*c)

    xmax = Einf/(m_e*c**2)

    integrand = lambda x: (
        x**(-Gamma)
        *
        (np.sqrt(x**2 + 1) - 1)
    )

    value, _ = quad(
        integrand,
        xinj,
        xmax
    )

    return value

In [17]:
def electron_equation(B):

    I1 = I1_of_B(B)

    rhs = (
        8*np.pi
        *
        (m_e*c**2)**(2-Gamma)
        *
        I1
        *
        J
    )

    return B**(alpha + 3) - rhs

In [18]:
sol = root_scalar(
    electron_equation,
    bracket=[1e-7, 1e-3]
)

B_e = sol.root

print("B_e =", B_e, "G")
print("B_e =", B_e*1e6, "microG")

B_e = 2.4122374898817357e-05 G
B_e = 24.122374898817355 microG


In [19]:
B0_e = kappa*B_e

print("B0_e =", B0_e*1e6, "microG")

B0_e = 7.346347195610718 microG


In [20]:
I1 = I1_of_B(B_e)

print("I1 =", I1)



print("pinj =", pinj)

print("xinj =", pinj/(m_e*c))

I1 = 3517.870414335114
pinj = 1.6332779950114298e-18
xinj = 0.05980677979191815


In [21]:
R = d*theta/2

print("R =", R, "cm")
print("R =", R/3.086e18, "pc")

R = 8.527078628542138e+19 cm
R = 27.631492639475496 pc


In [22]:
def I2_of_B(B):

    B0 = kappa*B

    pinj = xi*np.sqrt(2*m_p*k_B*Tp)

    Einf = (
    3*vs*e*B0*R
    /
    (8*c)
)
    xinj = pinj/(m_p*c)

    xmax = Einf/(m_p*c**2)

    integrand = lambda x: (
        x**(-Gamma)
        *
        (np.sqrt(x**2 + 1) - 1)
    )

    value, _ = quad(
        integrand,
        xinj,
        xmax
    )

    return value

In [23]:
def Kep():

    return (
        (m_e*Te)
        /
        (m_p*Tp)
    )**alpha

In [24]:
def proton_equation(B):

    I2 = I2_of_B(B)

    rhs = (
        8*np.pi
        *
        (m_p*c**2)**(2-Gamma)
        *
        I2
        *
        J
        /
        Kep()
    )

    return B**(alpha + 3) - rhs

In [25]:
sol_p = root_scalar(
    proton_equation,
    bracket=[1e-6, 1e-3]
)

B_p = sol_p.root

print("B_p =", B_p*1e6, "microG")

B_p = 77.38077763164827 microG


In [26]:
B0_p = kappa*B_p

print("B0_p =", B0_p*1e6, "microG")

B0_p = 23.56592421487927 microG


In [27]:
print("="*50)

print("Electron equipartition")

print("B_e  =", B_e*1e6,  "microG")
print("B0_e =", B0_e*1e6, "microG")

print()

print("Proton equipartition")

print("B_p  =", B_p*1e6,  "microG")
print("B0_p =", B0_p*1e6, "microG")

print("="*50)

Electron equipartition
B_e  = 24.122374898817355 microG
B0_e = 7.346347195610718 microG

Proton equipartition
B_p  = 77.38077763164827 microG
B0_p = 23.56592421487927 microG


In [28]:
Be_ref  = 20.0
B0e_ref = 6.1

Bp_ref  = 68.9
B0p_ref = 21.0

print("Electron B difference (%) =",
      100*(B_e*1e6 - Be_ref)/Be_ref)

print("Electron B0 difference (%) =",
      100*(B0_e*1e6 - B0e_ref)/B0e_ref)

print("Proton B difference (%) =",
      100*(B_p*1e6 - Bp_ref)/Bp_ref)

print("Proton B0 difference (%) =",
      100*(B0_p*1e6 - B0p_ref)/B0p_ref)

Electron B difference (%) = 20.611874494086777
Electron B0 difference (%) = 20.431921239519966
Proton B difference (%) = 12.308820945788483
Proton B0 difference (%) = 12.218686737520338


In [30]:
import pandas as pd

df = pd.DataFrame({
    "Quantity":[
        "B_e",
        "B0_e",
        "B_p",
        "B0_p"
    ],
    "Value (μG)":[
        B_e*1e6,
        B0_e*1e6,
        B_p*1e6,
        B0_p*1e6
    ]
})

print("\nFinal Results\n")
print(df.to_string(index=False))


Final Results

Quantity  Value (μG)
     B_e   24.122375
    B0_e    7.346347
     B_p   77.380778
    B0_p   23.565924
